In [0]:
#CSV files path
drivers_csv_path="/Volumes/workspace/default/rider_analytics_etl_pipeline/bronze/drivers_dw.csv"
trips_csv_path="/Volumes/workspace/default/rider_analytics_etl_pipeline/bronze/trips_dw.csv"
zones_csv_path="/Volumes/workspace/default/rider_analytics_etl_pipeline/bronze/zones_dw.csv"


In [0]:
#Storage paths
bronze_path="/Volumes/workspace/default/rider_analytics_etl_pipeline/bronze/"
silver_path="/Volumes/workspace/default/rider_analytics_etl_pipeline/silver/"
gold_path="/Volumes/workspace/default/rider_analytics_etl_pipeline/gold/"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import *

In [0]:
#Creating schemas
#trip_id	customer_id	driver_id	pickup_zone	drop_zone	pickup_datetime	drop_datetime	trip_distance	passenger_count
trips_schema = StructType([
               StructField("trip_id", IntegerType(), True),
               StructField("customer_id", IntegerType(), True),
               StructField("driver_id", IntegerType(), True),
               StructField("pickup_zone", IntegerType(),True),
               StructField("drop_zone", IntegerType(), True),
               StructField("pickup_datetime", TimestampType(), True),
               StructField("drop_datetime", TimestampType(), True),
               StructField("trip_distance", DoubleType(), True),
               StructField("passenger_count", IntegerType(), True)
               ])


#zone_id	zone_name	borough
zones_schema =StructType([
               StructField("zone_id", IntegerType(), True),
               StructField("zone_name",StringType(), True),
               StructField("borough", StringType(), True)
               ])

#driver_id	driver_name	vendor	experience_years
drivers_schema=StructType([
               StructField("driver_id", IntegerType(), True),
               StructField("driver_name", StringType(), True),
               StructField("vendor", IntegerType(), True),
               StructField("experience_years", IntegerType(),True)
               ])

In [0]:
#Reading trips.csv
trips_df= spark.read.csv(trips_csv_path, header=True, schema=trips_schema)
display(trips_df)

trip_id,customer_id,driver_id,pickup_zone,drop_zone,pickup_datetime,drop_datetime,trip_distance,passenger_count
1,145,13,20,16,2024-04-09T22:26:00.000Z,2024-04-09T22:31:00.000Z,16.82,3
2,59,91,8,7,2024-10-09T10:31:00.000Z,2024-10-09T11:27:00.000Z,12.8,2
3,232,11,10,15,2024-08-16T17:42:00.000Z,2024-08-16T17:53:00.000Z,1.63,3
4,24,76,11,14,2024-09-23T15:27:00.000Z,2024-09-23T15:40:00.000Z,4.34,2
5,225,94,17,17,2024-10-13T08:39:00.000Z,2024-10-13T09:18:00.000Z,18.36,2
6,242,124,10,11,2024-04-03T13:26:00.000Z,2024-04-03T14:23:00.000Z,16.28,4
7,194,58,13,18,2024-01-28T09:58:00.000Z,2024-01-28T10:12:00.000Z,7.95,4
8,138,32,15,12,2024-01-06T01:37:00.000Z,2024-01-06T01:58:00.000Z,13.78,3
9,211,96,4,8,2024-07-31T19:57:00.000Z,2024-07-31T20:26:00.000Z,9.96,3
10,57,17,15,10,2024-11-29T05:37:00.000Z,2024-11-29T06:21:00.000Z,13.34,1


In [0]:
#Reading zones.df
zones_df= spark.read.csv(zones_csv_path, header=True, schema=zones_schema)
display(zones_df)

zone_id,zone_name,borough
1,Zone_1,Brooklyn
2,Zone_2,Manhattan
3,Zone_3,Bronx
4,Zone_4,Queens
5,Zone_5,Queens
6,Zone_6,Bronx
7,Zone_7,Queens
8,Zone_8,Bronx
9,Zone_9,Queens
10,Zone_10,Brooklyn


In [0]:
#Reading drivers.csv
drivers_df= spark.read.csv(drivers_csv_path, header=True, schema=drivers_schema)
display(drivers_df)

driver_id,driver_name,vendor,experience_years
1,Driver_1,null,7
2,Driver_2,null,10
3,Driver_3,null,4
4,Driver_4,null,7
5,Driver_5,null,8
6,Driver_6,null,10
7,Driver_7,null,5
8,Driver_8,null,10
9,Driver_9,null,10
10,Driver_10,null,2


In [0]:
trips_df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(f"{bronze_path}/trips")


In [0]:
drivers_df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(f"{bronze_path}/drivers")

In [0]:
zones_df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(f"{bronze_path}/zones")

In [0]:
#trip_id	customer_id	driver_id	pickup_zone	drop_zone	pickup_datetime	drop_datetime
#zone_id	zone_name	borough
#driver_id	driver_name	vendor	experience_years

SILVER LAYER

In [0]:
silver_trips = spark.read.format("delta").load(f"{bronze_path}/trips")
silver_drivers = spark.read.format("delta").load(f"{bronze_path}/drivers")
silver_zones = spark.read.format("delta").load(f"{bronze_path}/zones")

In [0]:
silver_trips= silver_trips.dropna()

In [0]:
silver_trips = silver_trips.filter(col("trip_distance")>0)

In [0]:
silver_trips = silver_trips.toDF(*[c.lower() for c in trips_df.columns])
silver_zones = silver_zones.toDF(*[c.lower() for c in zones_df.columns])
silver_drivers = silver_drivers.toDF(*[c.lower() for c in drivers_df.columns])

In [0]:
# Remove exact duplicates
silver_trips = silver_trips.dropDuplicates()
silver_drivers = silver_drivers.dropDuplicates()
silver_zones = silver_zones.dropDuplicates()

In [0]:
#trip_id	customer_id	driver_id	pickup_zone	drop_zone	pickup_datetime	drop_datetime
#zone_id	zone_name	borough
#driver_id	driver_name	vendor	experience_years


In [0]:
trips_with_drivers_df = silver_trips.join(
    silver_drivers,
    silver_trips["driver_id"] == silver_drivers["driver_id"],
    "inner"
)

In [0]:
display(trips_with_drivers_df)

trip_id,customer_id,driver_id,pickup_zone,drop_zone,pickup_datetime,drop_datetime,trip_distance,passenger_count,driver_id,driver_name,vendor,experience_years
14,9,95,7,15,2024-12-07T21:34:00.000Z,2024-12-07T21:56:00.000Z,9.45,2,95,Driver_95,null,5
16,243,117,3,7,2024-03-10T02:34:00.000Z,2024-03-10T02:46:00.000Z,13.19,1,117,Driver_117,null,3
22,191,17,16,15,2024-07-10T13:00:00.000Z,2024-07-10T13:12:00.000Z,15.78,1,17,Driver_17,null,10
42,152,125,17,10,2024-09-03T22:21:00.000Z,2024-09-03T23:09:00.000Z,1.86,4,125,Driver_125,null,8
72,129,136,14,6,2024-10-10T07:21:00.000Z,2024-10-10T07:50:00.000Z,16.52,2,136,Driver_136,null,6
139,135,97,11,6,2024-08-29T02:37:00.000Z,2024-08-29T03:12:00.000Z,9.73,3,97,Driver_97,null,8
175,12,103,15,8,2024-03-30T02:31:00.000Z,2024-03-30T03:23:00.000Z,11.26,1,103,Driver_103,null,4
177,225,37,17,8,2024-03-27T06:05:00.000Z,2024-03-27T06:29:00.000Z,8.86,3,37,Driver_37,null,8
206,81,5,7,19,2024-03-06T07:13:00.000Z,2024-03-06T08:12:00.000Z,3.74,4,5,Driver_5,null,8
224,101,94,4,19,2024-10-24T06:38:00.000Z,2024-10-24T07:33:00.000Z,4.8,2,94,Driver_94,null,5


In [0]:
# Step 1
trips_with_drivers_df = silver_trips.join(
    silver_drivers,
    on="driver_id",
    how="inner"
)

# Step 2
trip_pickup_zone_df = trips_with_drivers_df.join(
    silver_zones,
    trips_with_drivers_df["pickup_zone"] == silver_zones["zone_id"],
    "inner"
).withColumnRenamed("zone_name", "pickup_zone_name") \
 .withColumnRenamed("borough", "pickup_borough")

# Step 3
drop_zones = silver_zones.withColumnRenamed("zone_id", "drop_zone_id") \
                        .withColumnRenamed("zone_name", "drop_zone_name") \
                        .withColumnRenamed("borough", "drop_borough")

# Step 4
trip_full_df = trip_pickup_zone_df.join(
    drop_zones,
    trip_pickup_zone_df["drop_zone"] == drop_zones["drop_zone_id"],
    "inner"
)

display(trip_full_df)

driver_id,trip_id,customer_id,pickup_zone,drop_zone,pickup_datetime,drop_datetime,trip_distance,passenger_count,driver_name,vendor,experience_years,zone_id,pickup_zone_name,pickup_borough,drop_zone_id,drop_zone_name,drop_borough
95,14,9,7,15,2024-12-07T21:34:00.000Z,2024-12-07T21:56:00.000Z,9.45,2,Driver_95,null,5,7,Zone_7,Queens,15,Zone_15,Bronx
117,16,243,3,7,2024-03-10T02:34:00.000Z,2024-03-10T02:46:00.000Z,13.19,1,Driver_117,null,3,3,Zone_3,Bronx,7,Zone_7,Queens
17,22,191,16,15,2024-07-10T13:00:00.000Z,2024-07-10T13:12:00.000Z,15.78,1,Driver_17,null,10,16,Zone_16,Queens,15,Zone_15,Bronx
125,42,152,17,10,2024-09-03T22:21:00.000Z,2024-09-03T23:09:00.000Z,1.86,4,Driver_125,null,8,17,Zone_17,Queens,10,Zone_10,Brooklyn
136,72,129,14,6,2024-10-10T07:21:00.000Z,2024-10-10T07:50:00.000Z,16.52,2,Driver_136,null,6,14,Zone_14,Brooklyn,6,Zone_6,Bronx
97,139,135,11,6,2024-08-29T02:37:00.000Z,2024-08-29T03:12:00.000Z,9.73,3,Driver_97,null,8,11,Zone_11,Bronx,6,Zone_6,Bronx
103,175,12,15,8,2024-03-30T02:31:00.000Z,2024-03-30T03:23:00.000Z,11.26,1,Driver_103,null,4,15,Zone_15,Bronx,8,Zone_8,Bronx
37,177,225,17,8,2024-03-27T06:05:00.000Z,2024-03-27T06:29:00.000Z,8.86,3,Driver_37,null,8,17,Zone_17,Queens,8,Zone_8,Bronx
5,206,81,7,19,2024-03-06T07:13:00.000Z,2024-03-06T08:12:00.000Z,3.74,4,Driver_5,null,8,7,Zone_7,Queens,19,Zone_19,Manhattan
94,224,101,4,19,2024-10-24T06:38:00.000Z,2024-10-24T07:33:00.000Z,4.8,2,Driver_94,null,5,4,Zone_4,Queens,19,Zone_19,Manhattan


In [0]:
gold_df = trip_full_df.select(
    "trip_id",
    "customer_id",
    "driver_id",
    "driver_name",
    "pickup_zone",
    "pickup_zone_name",
    "pickup_borough",
    "drop_zone",
    "drop_zone_name",
    "drop_borough",
    "trip_distance",
    "passenger_count"
)

In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/trip_analytics")

In [0]:
from pyspark.sql.functions import col

gold_df = gold_df.withColumn("fare", col("trip_distance") * 10)  # example logic

In [0]:
driver_performance_df = gold_df.groupBy("driver_id", "driver_name") \
    .agg(
        count("trip_id").alias("total_trips"),
        sum("fare").alias("total_earnings")
    )
display(driver_performance_df)

driver_id,driver_name,total_trips,total_earnings
95,Driver_95,10,1126.7
117,Driver_117,6,798.5999999999999
17,Driver_17,6,845.8000000000001
125,Driver_125,3,262.9
136,Driver_136,2,185.2
97,Driver_97,5,396.6
103,Driver_103,5,600.6
37,Driver_37,7,914.1
5,Driver_5,3,302.9
94,Driver_94,5,563.2


In [0]:
driver_performance_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/driver_performance")

In [0]:
pickup_zone_df = gold_df.groupBy("pickup_zone_name") \
    .count() \
    .withColumnRenamed("count", "total_trips") \
    .orderBy(col("total_trips").desc())

In [0]:
drop_zone_df = gold_df.groupBy("drop_zone_name") \
    .count() \
    .withColumnRenamed("count", "total_trips") \
    .orderBy(col("total_trips").desc())

In [0]:
pickup_zone_df.write.format("delta").mode("overwrite") \
    .save(f"{gold_path}/pickup_zone_analytics")

drop_zone_df.write.format("delta").mode("overwrite") \
    .save(f"{gold_path}/drop_zone_analytics")

In [0]:
revenue_city_df = gold_df.groupBy("pickup_borough") \
    .agg(
        sum("fare").alias("total_revenue")
    ) \
    .orderBy(col("total_revenue").desc())

In [0]:
revenue_city_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/revenue_by_city")